# STAT 764 · Lab 10 — Compress, and count who's missing

**Assigned Tue Nov 17 · Due Tue Dec 1, 11:59 pm · 25 points** (before or after Thanksgiving: your choice)

Individual work. About an hour; the whole notebook runs in seconds. You need Meeting 21 (PCA, inside a
pipeline, split by state) and Meeting 22 (missing data). Parts 1 and 2 can be done after Tuesday; Part 3 is easier
after Thursday.

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before submitting, upload
the `.ipynb` to Canvas.

**How it is graded.** Complete or incomplete, part by part. A part earns its points when everything it asks for is
in your notebook and has run. A written answer earns its points when it answers everything its part asks and points
at one of your numbers.

A state health department wants to use the County Health Rankings to (1) summarize each county in a few numbers and
(2) predict life expectancy from the other measures. You will use the **2024** file (Meeting 21 used 2025): how few
numbers carry most of the differences, whether a few components predict as well as all the measures, and which
counties any of this leaves out.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import numpy as np
import pandas as pd
from stat764 import load

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# County Health Rankings 2024: one row per county. The file's second row holds variable codes, so skip it; rows with
# County FIPS Code 0 are whole states and the nation, so drop them.
raw = load("county_health_2024.csv", low_memory=False, skiprows=[1])
raw = raw[pd.to_numeric(raw["County FIPS Code"], errors="coerce") != 0]

# Meeting 21's 39 county rates, less disability (not in the 2024 file) and air pollution (left out here): 37 measures.
MEASURES = [
    "Poor or Fair Health", "Poor Physical Health Days", "Poor Mental Health Days", "Frequent Physical Distress",
    "Frequent Mental Distress", "Diabetes Prevalence", "Adult Obesity", "Adult Smoking", "Physical Inactivity",
    "Excessive Drinking", "Insufficient Sleep", "Food Insecurity", "Uninsured", "Uninsured Adults",
    "Uninsured Children", "Flu Vaccinations", "Mammography Screening", "Preventable Hospital Stays",
    "High School Completion", "Some College", "Unemployment", "Children in Poverty", "Median Household Income",
    "Income Inequality", "Children in Single-Parent Households", "Social Associations", "Broadband Access",
    "Homeownership", "Severe Housing Problems", "Severe Housing Cost Burden", "Driving Alone to Work",
    "Long Commute - Driving Alone", "% Rural", "% 65 and Older", "% Below 18 Years of Age", "% Non-Hispanic White",
    "% Hispanic"]
OUTCOME = "Life Expectancy"

counties = raw[["State Abbreviation", "Name"]].rename(columns={"State Abbreviation": "state", "Name": "county"})
counties["population"] = pd.to_numeric(raw["Population raw value"], errors="coerce")   # for Part 3 only
for m in MEASURES + [OUTCOME]:
    counties[m] = pd.to_numeric(raw[f"{m} raw value"], errors="coerce")
counties = counties.reset_index(drop=True)

# Hold out a quarter of the STATES (all their counties) for the final test, as Meeting 21 grouped by state.
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=764)
                           .split(counties, groups=counties["state"]))
train, test = counties.iloc[train_idx], counties.iloc[test_idx]

# Parts 1 and 2 use only the counties with all 37 measures and life expectancy.
complete_train = train.dropna(subset=MEASURES + [OUTCOME])
complete_test = test.dropna(subset=MEASURES + [OUTCOME])
print(f"{len(counties):,} counties · training states {train['state'].nunique()} ({len(train):,} counties,"
      f" {len(complete_train):,} complete) · test states {test['state'].nunique()} ({len(test):,} counties,"
      f" {len(complete_test):,} complete)")

---

## Part 1 · How few numbers carry the differences? (5 points)

Use `complete_train` only.

**(i)** Standardize the 37 measures and fit `PCA()`: `pca = PCA().fit(StandardScaler().fit_transform(
complete_train[MEASURES]))`. Print the share of the variance of components 1 to 5, and the number of components
needed for 80% (`np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.80) + 1`).

**(ii)** Print component 1's 6 largest loadings **by size** (positive or negative):
`pd.Series(pca.components_[0], index=MEASURES).sort_values(key=abs, ascending=False).head(6)`.

In [ ]:
# YOUR CODE HERE

## Part 2 · Choose how many components, then test (5 points)

Choosing is fitting (Meeting 8): the best of several cross-validated scores is too optimistic, so it is the number
you choose by, not the number you report. Choose using the **training states only**; then score the test states once.

**(i)** For every k from 1 to 37, the cross-validated RMSE of
`make_pipeline(StandardScaler(), PCA(n_components=k), LinearRegression())` on `complete_train`, with
`GroupKFold(n_splits=5)` and `groups=complete_train["state"]`. (`cross_val_score` with
`scoring="neg_root_mean_squared_error"` returns minus the RMSE.) Print the best k and its RMSE, and the smallest k
whose RMSE is within 0.1 years of the best.

**(ii)** Fit two pipelines on all of `complete_train`: the best k's, and `make_pipeline(StandardScaler(),
LinearRegression())` on all 37 measures. Print each one's RMSE on `complete_test`
(`root_mean_squared_error(complete_test[OUTCOME], model.predict(complete_test[MEASURES]))`).

In [ ]:
# YOUR CODE HERE

## Part 3 · Who is missing, and what to do about it (5 points)

Parts 1 and 2 quietly dropped every county missing a measure or life expectancy.

**(i)** Across all counties (`counties`): how many lack at least one of the 37 measures, and how many lack life
expectancy? Then one small table comparing the counties **with** all 37 measures to those **without**: the number
of counties, the median `population`, and the median `% Rural` (a share from 0 to 1).

**(ii)** Two ways to predict life expectancy, both fitted on the training counties that **have** life expectancy:

- **A, drop:** `make_pipeline(StandardScaler(), LinearRegression())`, fitted on the training counties with all 37
  measures.
- **B, impute:** `make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
  LinearRegression())`, fitted on every training county with life expectancy, missing measures and all. The imputer
  fills each gap with the training median and adds a 0/1 "was missing" column for each measure that had gaps.

On the test counties with life expectancy, print: how many each model can predict for (A needs all 37 measures),
each one's RMSE on the counties it can predict for, B's RMSE on the same counties as A, and B's RMSE on the test
counties A cannot predict for.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written: for the health department (10 points)

Plain words, and one number from your output in each answer.

**(a)** (3 points) Name component 1 from its largest loadings. Is it the same direction Meeting 21 found in the
2025 data, and how much of the variance does it carry here?

**(b)** (2 points) Which k won on the training states, and how did it do on the test states against all 37
measures? Does compressing pay for this prediction?

**(c)** (3 points) Which counties are missing data, and once they are dropped, which counties can the model speak
for? Compare A and B on the same counties: what does imputing buy, and what does it risk?

**(d)** (2 points) The department wants to rank counties by their component 1 score. Name two things to tell them
first: one from Part 3 (which counties would get no score), and one from Parts 1–2 or Meeting 21.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 prints the shares of components 1–5, the number for 80%, and component 1's six largest loadings by size
- [ ] Part 2 prints the best k with its cross-validated RMSE, the smallest k within 0.1 years, and both test RMSEs
- [ ] Part 3 prints the missing counts, the with/without table, and for A and B the counties predicted and the RMSEs
  (B on its own counties, on A's, and on the ones A cannot predict)
- [ ] All four written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**25 points.**